# ⚡ Load-Shedding Risk Early Warning for Bangladesh — full pipeline on Kaggle

**Team:** Swagoto Utsab Singha Roy (23101124) · Lamiya Zaman (23101132) · Jannatul Ferdoushi Islam (23101119)
Department of CSE, University of Asia Pacific · Machine Learning Lab project

This notebook runs the **whole project end to end** and shows every result:

| step | what happens | time (Kaggle CPU) |
|---|---|---|
| 0 | find the project, install `holidays` if missing, check the saved models | < 1 min |
| 1 | leakage / correctness tests | < 1 min |
| 2 | build labels + 14-day windows for 10 units (national + 9 zones) | < 1 min |
| 3 | baselines: persistence, BPDB-forecast rule, LogReg, RandomForest, XGBoost + SHAP | ~1 min |
| 4 | **multimodal GRU** with modality attention (3 seeds), temperature scaling, abstain | 5–10 min |
| 5 | modality ablation + leave-one-division-out | 15–25 min (skip with `RUN_LONG=False`) |
| 6 | final validation-tuned ensemble, refit, test 2025–2026 once, live forecast | 10–15 min |
| 7 | **lab experiments**: PR-AUC, attention vs concatenation, multi- vs single-task, walk-forward | 10–40 min |
| 8 | demo: "time machine" forecast for any past day + final alert | seconds |

**How to get the code here (pick one):**
* **A — GitHub (easiest):** Settings → *Internet* ON (needs a phone-verified Kaggle account), then just run the cells. Put your repo URL in the next cell.
* **B — Kaggle Dataset:** download the repo as a ZIP, *Create → New Dataset* → upload the ZIP, then in this notebook *Add Input* → your dataset. The cell below finds it automatically.

Set `QUICK = True` for a 10-minute demo run (1 seed, 10 epochs). `QUICK = False` reproduces the report numbers.

In [ ]:
# ---------------- settings ----------------
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/loadshed-early-warning.git"   # <- change to your repo
QUICK    = False   # True = fast demo (1 seed, 10 epochs). False = full run (same numbers as the report)
RUN_LONG = True    # False skips step 5 (ablation + leave-one-division-out), the slowest step
RETRAIN  = True    # False = use the saved trained models only (no training at all, everything runs in ~2 min)

In [ ]:
import os, sys, glob, shutil, subprocess
from pathlib import Path

def find_project():
    here = Path.cwd()
    for p in [here, here.parent]:                       # running locally from the repo or repo/notebooks
        if (p / "src" / "config.py").exists():
            return p
    work = Path("/kaggle/working/loadshed-early-warning")
    if (work / "src" / "config.py").exists():
        return work
    hits = glob.glob("/kaggle/input/**/src/config.py", recursive=True)   # option B: attached Kaggle dataset
    if hits:
        shutil.copytree(Path(hits[0]).parents[1], work)                  # inputs are read-only -> copy
        return work
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)], check=True)   # option A: GitHub
    return work

ROOT = find_project()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# holidays is the only library Kaggle may not have pre-installed
try:
    import holidays
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
import numpy as np, pandas as pd, torch, xgboost, sklearn, shap
print("python", sys.version.split()[0], "| torch", torch.__version__, "| xgboost", xgboost.__version__,
      "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
pd.set_option("display.width", 220, "display.max_columns", 40)
from IPython.display import Image, display, Markdown
FIG, TAB = ROOT / "outputs" / "figures", ROOT / "outputs" / "tables"
def show(*names, width=900):
    for n in names:
        f = FIG / n
        if f.exists(): display(Image(str(f), width=width))
        else: print("missing figure:", n)
def run(script, *args):
    r = subprocess.run([sys.executable, f"scripts/{script}", *args], capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-3000:] if r.returncode else "")
    if r.returncode: raise RuntimeError(f"{script} failed")
Q = ["--quick"] if QUICK else []

## Step 0 — are the saved models the ones behind the reported numbers?
The repository ships the trained weights. This re-computes their test predictions and compares with the stored ones (difference should be ~1e-6).

In [ ]:
run("check_saved_models.py")

## Step 1 — leakage tests
1) chronological split boundaries, 2) 300 random labels recomputed from raw data, 3) **future-perturbation test**: every value after the validation period is scrambled; train/validation features, labels and thresholds must not change, 4) no NaN reaches the model.

In [ ]:
r = subprocess.run([sys.executable, "tests/test_pipeline.py"], capture_output=True, text=True); print(r.stdout, r.stderr[-2000:])

## Step 2 — data → labels → 14-day windows
Low = no shedding; Medium = shedding/demand ≤ threshold; High = above. Threshold = median of the non-zero ratios in the **training years only** (7.6 % for zones, 1.9 % for the national grid).

In [ ]:
run("01_prepare.py")
show("eda_demand_shedding.png", "eda_class_counts.png")
pd.read_csv(TAB / "class_counts_by_year_T+1.csv")

## Step 3 — baselines (persistence, BPDB-forecast rule, Logistic Regression, Random Forest, XGBoost) + SHAP

In [ ]:
if RETRAIN: run("03_baselines.py")
b = pd.read_csv(TAB / "baselines_test_metrics.csv")
display(b[b.group == "All"][["model", "horizon", "macro_f1", "high_recall", "high_precision", "missed_high", "false_alarm"]].round(3))
show("shap_top_T+1.png")

## Step 4 — the multimodal GRU (grid GRU + weather GRU + context MLP → modality attention → T+1 / T+3 heads)
Three seeds, class-weighted cross-entropy, AdamW, early stopping on validation macro-F1; then temperature scaling and the abstain threshold τ, both fitted on validation.

In [ ]:
if RETRAIN: run("04_train_nn.py", *Q)
display(pd.read_csv(TAB / "nn_test_metrics_mean_std.csv", header=[0, 1], index_col=[0, 1]).round(3))
show("calibration_reliability.png", "confusion_mmgru.png", "modality_attention.png", "abstain_coverage_curve.png", width=700)

## Step 5 — modality ablation and leave-one-division-out (slow)

In [ ]:
if RETRAIN and RUN_LONG: run("05_ablation_lodo.py", *Q)
display(pd.read_csv(TAB / "ablation_modalities.csv")[["model", "horizon", "macro_f1", "high_recall", "missed_high", "false_alarm"]].round(3))
show("ablation_modalities.png", "lodo_T+1.png", width=700)

## Step 6 — final system: validation-tuned ensemble (GRU + XGBoost + persistence), refit, one honest test on 2025–2026, live forecast

In [ ]:
if RETRAIN: run("09_final_pipeline.py")
fc = pd.read_csv(TAB / "final_comparison.csv")
display(fc[["model", "horizon", "macro_f1", "bal_acc", "high_recall", "high_precision", "missed_high", "false_alarm", "ece", "coverage"]].round(3))
show("final_comparison.png")

## Step 7 — ML-lab experiments promised in the proposal
(a) PR-AUC of every model · (b) attention vs concatenation fusion · (c) multi-task vs single-task · (d) walk-forward (expanding-window) validation 2023 → 2026.

In [ ]:
if RETRAIN: run("10_lab_experiments.py", *Q)
display(pd.read_csv(TAB / "lab_pr_auc.csv")[["model", "horizon", "macro_f1", "pr_auc_macro", "pr_auc_high", "roc_auc_macro"]].round(3))
display(pd.read_csv(TAB / "lab_fusion_task_summary.csv").round(3))
wf = pd.read_csv(TAB / "lab_walk_forward.csv")
display(wf.pivot_table(index=["horizon", "model"], columns="test_year", values="macro_f1").round(3))
show("lab_pr_curves.png", "lab_fusion_task.png", "lab_walk_forward.png", width=800)

## Step 8 — demo: run the trained model on any day ("time machine")
Pick a past date: the 3 saved GRU seeds read the 14 days before it and predict T+1 / T+3; the table also shows what really happened.

In [ ]:
AS_OF = "2026-07-15"     # <- try any date between 2019-12-04 and 2026-10-28
run("08_forecast_next_days.py", "--as-of", AS_OF)
display(pd.read_csv(TAB / f"forecast_{AS_OF}.csv"))
display(Image(str(FIG / f"forecast_{AS_OF}.png"), width=650))

## Final alert of the deployed ensemble (data up to the newest BPDB report)

In [ ]:
f = sorted(glob.glob(str(TAB / "forecast_final_*.csv")))[-1]
display(pd.read_csv(f)); display(Image(f.replace("tables", "figures").replace(".csv", ".png"), width=700))

In [ ]:
# summary of everything + a zip of all outputs (download it from the Output panel on the right)
run("07_report.py")
shutil.make_archive("/kaggle/working/loadshed_outputs" if Path("/kaggle/working").exists() else str(ROOT / "loadshed_outputs"), "zip", ROOT / "outputs")
display(Markdown((ROOT / "outputs" / "summary.md").read_text(encoding="utf-8")[:6000]))